## Insertion/Deletion Curve — Single-Sample Demo

Evaluating FIxLIP attribution quality via deletion curves and AID.

In [1]:
import torch
torch.cuda.empty_cache()

In [2]:
import torch
from transformers import CLIPProcessor, CLIPModel
from PIL import Image
import matplotlib.pyplot as plt
import numpy as np
import shapiq
import src
from ImputerFactory import ImageImputerFactory
from Game import VisionLanguageGame

### 1. Load model & data

In [3]:
model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
model.to('cuda')
processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


In [4]:
input_text = "black dog next to a yellow hydrant"
input_image = Image.open("assets/dog_and_hydrant.png")

### 2. Build Imputer & Game

In [5]:
factory = ImageImputerFactory()
imputer = factory.build(model, processor, input_image, input_text)
game = VisionLanguageGame(imputer, batch_size=64)
n_total = game.n_players_image + game.n_players_text
print(f"n_img={game.n_players_image}, n_txt={game.n_players_text}, n_total={n_total}")

n_img=49, n_txt=8, n_total=57


d:\anaconda3\envs\fixlip\lib\site-packages\transformers\models\clip\modeling_clip.py:546: UserWarning: 1Torch was not compiled with flash attention. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\builder\windows\pytorch\aten\src\ATen\native\transformers\cuda\sdp_utils.cpp:555.)
  attn_output = torch.nn.functional.scaled_dot_product_attention(


In [ ]:
# ── Build old-style game (src pipeline) for comparison ──
import src.game_huggingface
game_old = src.game_huggingface.VisionLanguageGame(
    model, processor,
    input_image=input_image,
    input_text=input_text,
    batch_size=64,
)

print(f"Old: n_img={game_old.n_players_image}, n_txt={game_old.n_players_text}, n_total={game_old.n_players}")
print(f"New: n_img={game.n_players_image}, n_txt={game.n_players_text}, n_total={game.n_players}")

### 3. Compute FIxLIP interaction values

In [6]:
src.utils.set_seed(0)
fixlip = src.fixlip.FIxLIP(
    n_players_image=game.n_players_image,
    n_players_text=game.n_players_text,
    max_order=2,
    p=0.5,
    mode="banzhaf",
    random_state=0,
)
interaction_values = fixlip.approximate_crossmodal(game, budget=2**18)
print(interaction_values)

InteractionValues(
    index=FWBII, max_order=2, min_order=0, estimated=True, estimation_budget=262080,
    n_players=57, baseline_value=0.047953978272793615,
    Top 10 interactions:
        (54,): 2.338390111094565
        (33, 55): 1.714555922819816
        (55, 56): 1.3486586080815728
        (28, 50): 1.3138865232805506
        (29, 50): 1.3001619973733363
        (36, 50): 1.2252881915976555
        (52,): -2.274011876884654
        (56,): -2.31028291370796
        (50,): -4.703247396860848
        (55,): -6.7424567324005205
)


### 4. Run Deletion Curves (MIF & LIF)

In [13]:
# Extract 1st-order attribution values
first_order = src.utils.convert_iv_to_first_order(interaction_values)
attr = first_order.get_n_order(1).values  # shape (n_total,)
attr_sorted = np.sort(attr)  # ascending

# MIF (Most Important First): remove high-attr → low-attr
# threshold descends from max → min
# attr <= max = all True (full coalition)
# attr <= min = only the single minimum-attr player is True (near empty)
coalition_mif = np.stack(
    [attr <= v for v in attr_sorted[::-1]] + [np.zeros(n_total, dtype=bool)]
)
pred_mif = game.value_function(coalition_mif)

# LIF (Least Important First): remove low-attr → high-attr  
# threshold ascends from min → max
# attr >= min = all True (full coalition)
# attr >= max = only the single maximum-attr player is True (near empty)
coalition_lif = np.stack(
    [attr >= v for v in attr_sorted] + [np.zeros(n_total, dtype=bool)]
)
pred_lif = game.value_function(coalition_lif)

print(f"MIF: {pred_mif[0]:.4f} (full) → {pred_mif[-1]:.4f} (empty)")
print(f"LIF: {pred_lif[0]:.4f} (full) → {pred_lif[-1]:.4f} (empty)")

MIF: 33.9476 (full) → 22.8664 (empty)
LIF: 33.9476 (full) → 22.8664 (empty)


### 5. Normalize & compute AID

In [ ]:
# Normalize both curves to [0, 1] using global min/max across BOTH curves
# This ensures no value exceeds [0, 1] — fixes the "LIF > 1" issue
v_min = min(pred_mif.min(), pred_lif.min())
v_max = max(pred_mif.max(), pred_lif.max())
print(f"v_min={v_min:.4f}, v_max={v_max:.4f}, range={v_max - v_min:.4f}")

mif_norm = (pred_mif - v_min) / (v_max - v_min)
lif_norm = (pred_lif - v_min) / (v_max - v_min)

# AID = mean area between LIF and MIF
aid = np.mean(lif_norm - mif_norm)
print(f"AID = {aid:.4f}  (0=random, larger=better, max≈1)")
print(f"mif_norm range: [{mif_norm.min():.4f}, {mif_norm.max():.4f}]")
print(f"lif_norm range: [{lif_norm.min():.4f}, {lif_norm.max():.4f}]")

### 6. Plot

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Left: raw similarity curves ---
ax = axes[0]
x = np.arange(len(pred_mif))
ax.plot(x, pred_mif, 'r-', linewidth=2, label='MIF (Most Important First)')
ax.plot(x, pred_lif, 'b-', linewidth=2, label='LIF (Least Important First)')
ax.set_xlabel('Features removed')
ax.set_ylabel('CLIP similarity')
ax.set_title(f'Deletion Curves (raw)\nempty={v_min:.2f}, full={v_max:.2f}')
ax.legend()

# --- Right: normalized curves + AID fill ---
ax = axes[1]
x_frac = np.arange(len(mif_norm)) / (len(mif_norm) - 1)
ax.plot(x_frac, mif_norm, 'r-', linewidth=2, label='MIF')
ax.plot(x_frac, lif_norm, 'b-', linewidth=2, label='LIF')

# Fill only where LIF > MIF (typical case), otherwise fill with red warning
ax.fill_between(x_frac, lif_norm, mif_norm, where=(lif_norm >= mif_norm),
                alpha=0.15, color='green', label=f'AID={aid:.3f}')
ax.fill_between(x_frac, lif_norm, mif_norm, where=(lif_norm < mif_norm),
                alpha=0.15, color='red', label='MIF > LIF (bad)')

ax.set_xlabel('Fraction of features removed')
ax.set_ylabel('Normalized similarity')
ax.set_title('Deletion Curves (normalized)')
ax.legend()
ax.set_xlim(-0.02, 1.02)
ax.set_ylim(-0.02, 1.02)  # give 2% headroom for curves touching edges

plt.tight_layout()
plt.show()

### Interpretation

- **MIF curve（红线）**：从完整输入开始，优先删最重要的特征。如果曲线从 0 开始就快速下跌 → 归因准确。
- **LIF curve（蓝线）**：从完整输入开始，优先删最不重要的特征。如果曲线保持高位直到最后才跌 → 归因准确。
- **AID（绿色区域）**：LIF − MIF 的平均面积。正值越大越好。
- **边界条件**：两条曲线最终在 empty coalition（-0.92 附近）汇合。

### 7. Numerical Validation: Old vs New Pipeline

Compare value_function outputs between old (`src.game_huggingface`) and new (`ImputerFactory + Game`) pipelines on the same coalitions.

In [ ]:
# 1. Check player counts match
assert game_old.n_players == game.n_players, \
    f"Player count mismatch: old={game_old.n_players}, new={game.n_players}"
print("Player counts match:", game_old.n_players)

# 2. Compare empty / full coalition values
empty_old = game_old.empty_coalition_value
empty_new = game.normalization_value
print(f"Empty:  old={empty_old:.6f}, new={empty_new:.6f}, diff={abs(empty_old - empty_new):.2e}")

# Both pipelines should produce identically structured empty coalition
co_full = np.ones((1, game.n_players), dtype=bool)  # full coalition
co_empty = np.zeros((1, game.n_players), dtype=bool)  # empty coalition

full_old = game_old.value_function(co_full)[0]
full_new = game.value_function(co_full)[0]
empty_v_old = game_old.value_function(co_empty)[0]
empty_v_new = game.value_function(co_empty)[0]

print(f"Full:  old={full_old:.6f}, new={full_new:.6f}, diff={abs(full_old - full_new):.2e}")
print(f"Empty: old={empty_v_old:.6f}, new={empty_v_new:.6f}, diff={abs(empty_v_old - empty_v_new):.2e}")

# 3. Compare random coalitions
np.random.seed(42)
n_test = 200
test_coalitions = np.random.rand(n_test, game.n_players) > 0.5

values_old = game_old.value_function(test_coalitions)
values_new = game.value_function(test_coalitions)

diff = np.abs(values_old - values_new)
max_diff = diff.max()
mean_diff = diff.mean()
print(f"\n{n_test} random coalitions:")
print(f"  max_diff  = {max_diff:.6e}")
print(f"  mean_diff = {mean_diff:.6e}")

if max_diff < 1e-4:
    print("  → PASS (max_diff < 1e-4)")
elif max_diff < 1e-3:
    print(f"  → CLOSE (max_diff = {max_diff:.2e}, within 1e-3)")
else:
    print(f"  → FAIL (max_diff = {max_diff:.6e} > 1e-4)")
    # Show the worst coalition
    worst_idx = diff.argmax()
    print(f"  Worst coalition index: {worst_idx}")
    print(f"  old={values_old[worst_idx]:.6f}, new={values_new[worst_idx]:.6f}")